# Experiment 3: **Performance on Different Games**

## Objective
In this experiment, we compute the evaluation metrics for different models across multiple games. We extend the analysis by comparing model performance on all games provided by the original paper.

---

## Methodology
- **Models Used:**  
  - GPT-4o Mini  
  - Qwen2.5-72B (int4)  
  - Mistral-Small  

- **Games Evaluated:**  
  - **Base**  
  - **Base Rewritten**  
  - **Game1**  
  - **Game2**  
  - **Game3**  

- **Evaluation Metrics:**  
  The script will compute the following metrics for each model and game combination:  
  - **% 5/6-way agreement**  
  - **% 6-way agreement**  
  - **% Any agreement**  

---

## Results
The results of this experiment will provide insights into how each model performs across different games, highlighting their strengths and weaknesses in terms of agreement rates. These findings are presented in **Table 2** of our paper.

In [4]:
import os
import eval_utils as evaluation
import json
import numpy as np
import pandas as pd
from IPython.display import display

raw_path = '../our_games_descriptions'

games = ['base', 'base_rewritten', 'game1', 'game2', 'game3']
models = ['gpt4o-mini', 'Mistral-Small-Instruct-2409', 'Qwen2.5-72B-Instruct']

results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6

for game in games:
    for model in models:

        directory = os.path.join(raw_path, game, 'output', 'original_code', model)
        agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
        answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

        num_rounds = 0
        for file_ in answers_files:
            answers = json.load(open(file_))
            _num_rounds = len(answers['rounds'])
            num_rounds = max(num_rounds, _num_rounds)

        # Track statistics
        feasible_in_last_step = 0
        accepted_by_all_in_last_step = 0
        contained_feasible_deal = 0
        successfull_games = 0
        total_rounds = 0

        # Loop through all answer files (each represents a game)
        for file_ in answers_files:
            answers = json.load(open(file_))
            
            if len(answers['rounds']) != num_rounds:
                print(f"WARNING: Game {file_} has a different number of rounds")
                continue
            total_rounds += len(answers['rounds'])
            successfull_games += 1

            # Extract deals for this game
            feasible_found = False

            # Extract the name of the first player (p1) to validate feasibility throughout the game
            p1_name = answers['rounds'][0]['agent']

            total_deals = 0
            
            for i, round_ in enumerate(answers['rounds']):
                name, answer = round_['agent'], round_['public_answer']
                deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

                try:
                    deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
                except:
                    print(f"Error in game {file_} round {i}")
                    continue

                if issues_suggested >= ISSUES_NUM:
                    total_deals += 1

                # Check if the deal was feasible at any point (Deal must have been proposed by p1)
                if evaluation.is_feasible(agents, deal) and name == p1_name:
                    feasible_found = True
            

            # CHECK GAME COMPLETION METRICS

            last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
            
            # 1. Check if the last deal is feasible
            if evaluation.is_feasible(agents, last_deal):
                feasible_in_last_step += 1

            # 2. Check if the last deal is acceptable by all agents
            all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
            if all_accept:
                accepted_by_all_in_last_step += 1

            # 3. Check if any deal during the game was in the feasibility set
            if feasible_found:
                contained_feasible_deal += 1

        # Compute percentages
        num_games = successfull_games
        perc_feasible_last = (feasible_in_last_step / num_games) * 100
        perc_accepted_all_last = (accepted_by_all_in_last_step / num_games) * 100
        perc_feasible_any = (contained_feasible_deal / num_games) * 100

        results[(game, model)] = {
            "5/6-way (%)": f"{perc_feasible_last:.2f}",
            "6-way (%)": f"{perc_accepted_all_last:.2f}",
            "Any (%)": f"{perc_feasible_any:.2f}",
        }

df = pd.DataFrame(results).T

# Initialize table storage
table_data = []

# Iterate over games and organize metrics side by side for each model
for game in games:
    row = [game]  # Start row with the game name
    
    for metric in ["5/6-way (%)", "6-way (%)", "Any (%)"]:
        metric_values = [results[(game, model)][metric] for model in models]
        row.append(" / ".join(metric_values))  # Format as "model1 / model2 / model3"
    
    table_data.append(row)

# Define column names
columns = ["Game", "5/6-way", "6-way", "Any"]

# Convert to DataFrame
df_final = pd.DataFrame(table_data, columns=columns)

# Display nicely formatted table
df_styled = df_final.style.set_properties(**{"text-align": "center"}) \
                          .set_caption("Game Performance Across Models") \
                          .set_table_styles([
                              {'selector': 'th', 'props': [('font-size', '14px'), ('text-align', 'center')]},
                              {'selector': 'td', 'props': [('font-size', '13px')]}
                          ])

display(df_styled)

,Game,5/6-way,6-way,Any
0,base,55.00 / 80.00 / 85.00,5.00 / 0.00 / 0.00,90.00 / 100.00 / 95.00
1,base_rewritten,55.00 / 60.00 / 95.00,0.00 / 15.00 / 5.00,80.00 / 95.00 / 100.00
2,game1,35.00 / 25.00 / 80.00,5.00 / 5.00 / 30.00,75.00 / 75.00 / 95.00
3,game2,15.00 / 40.00 / 30.00,15.00 / 40.00 / 15.00,30.00 / 85.00 / 60.00
4,game3,55.00 / 0.00 / 80.00,55.00 / 0.00 / 80.00,95.00 / 50.00 / 100.00
